In [ ]:
# ================================================================
# USER CONFIGURATION — set this path for your environment
# ================================================================
MAPS_SAVE_DIR = ''    # e.g. '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
# ================================================================

# 05 — Analysis and Visualisation

This notebook produces all figures and tables for the thesis results section.
No model training occurs here — all results are loaded from CSV files
saved by the experiment notebooks.

**Run order:**
1. `02_standard_protocol.ipynb` - standard protocol results
2. `03_crossview_protocol.ipynb` - cross-view protocol results
3. `04_ablation_study.ipynb` - ablation investigations
4. `06_mvtec_validation.ipynb` - AnomalyDINO MVTec validation
5. This notebook - analysis, figures, tables

**Thesis results structure:**
- Standard Protocol: aggregate metrics, per-category, per-viewpoint, per-defect-type, WGA, disagreement
- Cross-View Protocol: aggregate metrics, per-viewpoint delta, degradation ratios
- Ablation Investigations 1-4
- MVTec AnomalyDINO Validation
- Appendix raw tables

## 0. Infrastructure

In [ ]:
import os
import sys
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

repo_path     = '/content/drive/MyDrive/BachelorsThesis'
results_path  = f'{repo_path}/results'
figures_path  = f'{repo_path}/results/figures'
MAPS_SAVE_DIR = f'{results_path}/anomaly_maps'

os.makedirs(figures_path, exist_ok=True)

if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
else:
    !git -C {repo_path} pull

sys.path.insert(0, repo_path)
!pip install anomalib==2.3.3 ADEval einops timm kornia -q

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

metrics    = load_module('metrics', f'{repo_path}/evaluation/metrics.py')
wga_module = load_module('wga',     f'{repo_path}/evaluation/wga.py')

compute_i_auroc           = metrics.compute_i_auroc
compute_s_auroc           = metrics.compute_s_auroc
compute_all_metrics       = metrics.compute_all_metrics
compute_degradation_ratio = metrics.compute_degradation_ratio
wga_by_category           = wga_module.wga_by_category
wga_by_viewpoint          = wga_module.wga_by_viewpoint
wga_by_defect_type        = wga_module.wga_by_defect_type
print_wga_summary         = wga_module.print_wga_summary
find_disagreement_groups  = wga_module.find_disagreement_groups

print('Infrastructure ready')
print(f'Repo:    {repo_path}')
print(f'Results: {results_path}')
print(f'Figures: {figures_path}')
print(f'Maps:    {MAPS_SAVE_DIR}')

In [ ]:
import zipfile, shutil, os

zip_dir    = '/content/drive/MyDrive/datasets/realiad_512/realiad_512'
target_dir = '/content/realiad_512'
os.makedirs(target_dir, exist_ok=True)

json_src = '/content/drive/MyDrive/datasets/realiad_512/realiad_jsons'
json_dst = f'{target_dir}/realiad_jsons'
if not os.path.exists(json_dst):
    shutil.copytree(json_src, json_dst)
    print('JSONs copied')
else:
    print('JSONs already present')

for f in sorted(os.listdir(zip_dir)):
    if f.endswith('.zip'):
        category = f.replace('.zip', '')
        if not os.path.exists(f'{target_dir}/{category}'):
            print(f'Unzipping {f}...')
            with zipfile.ZipFile(f'{zip_dir}/{f}', 'r') as z:
                z.extractall(target_dir)
        else:
            print(f'Skipping {category} — already present')

print(f'Dataset ready at: {target_dir}')

## 1. Results Paths

In [ ]:
std_results  = f'{results_path}/standard'
cv_results   = f'{results_path}/crossview'
abl_results  = f'{results_path}/ablation'

std_maps     = f'{MAPS_SAVE_DIR}/standard'
cv_maps      = f'{MAPS_SAVE_DIR}/crossview'

maps_abl1_mm = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_multiview'
maps_abl1_sm = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_multiview'
maps_abl1_ms = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_singleview'
maps_abl1_ss = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_singleview'
maps_abl4    = f'{MAPS_SAVE_DIR}/ablation/investigation4'

print('Paths set')

## 2. Load All Results

In [ ]:
def safe_read_csv(path, label):
    """Load CSV if it exists, return None and print warning if missing."""
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f'  OK       {label} ({len(df)} rows)')
        return df
    else:
        print(f'  MISSING  {label}')
        print(f'           {path}')
        return None

print('=== Standard Protocol ===')
results_din_std  = safe_read_csv(f'{std_results}/dinomaly_scores.csv',    'Dinomaly standard')
results_dino_std = safe_read_csv(f'{std_results}/anomalydino_scores.csv', 'AnomalyDINO standard')
results_inp_std  = safe_read_csv(f'{std_results}/inpformer_scores.csv',   'INP-Former standard')

print('\n=== Cross-View Protocol ===')
results_din_cv   = safe_read_csv(f'{cv_results}/dinomaly_scores.csv',    'Dinomaly cross-view')
results_dino_cv  = safe_read_csv(f'{cv_results}/anomalydino_scores.csv', 'AnomalyDINO cross-view')
results_inp_cv   = safe_read_csv(f'{cv_results}/inpformer_scores.csv',   'INP-Former cross-view')

print('\n=== Investigation 1 — Factorial ===')
abl1_cond_a = safe_read_csv(
    f'{std_results}/anomalydino_scores.csv',
    'Cond A (MC+MV)')
abl1_cond_b = safe_read_csv(
    f'{abl_results}/investigation1/anomalydino_singleclass_multiview_scores.csv',
    'Cond B (SC+MV)')
abl1_cond_c = safe_read_csv(
    f'{abl_results}/investigation1/anomalydino_multiclass_singleview_scores.csv',
    'Cond C (MC+SV)')
abl1_cond_d = safe_read_csv(
    f'{abl_results}/investigation1/anomalydino_singleclass_singleview_scores.csv',
    'Cond D (SC+SV)')

print('\n=== Investigation 2 — Compute Equalisation ===')
abl2_inp_eq = safe_read_csv(
    f'{abl_results}/investigation2/inpformer_equalised_scores.csv',
    'INP-Former 22 epoch')

print('\n=== Investigation 3 — Volume Compensation ===')
abl3_inp_comp = safe_read_csv(
    f'{abl_results}/investigation3/inpformer_cv_compensated_scores.csv',
    'INP-Former CV compensated')

print('\n=== Investigation 4 — Category Scaling ===')
abl4_din_10 = safe_read_csv(
    f'{abl_results}/investigation4/real_iad/dinomaly_10cat_scores.csv',    'Dinomaly 10-cat')
abl4_din_20 = safe_read_csv(
    f'{abl_results}/investigation4/real_iad/dinomaly_20cat_scores.csv',    'Dinomaly 20-cat')
abl4_inp_10 = safe_read_csv(
    f'{abl_results}/investigation4/real_iad/inpformer_10cat_scores.csv',   'INP-Former 10-cat')
abl4_inp_20 = safe_read_csv(
    f'{abl_results}/investigation4/real_iad/inpformer_20cat_scores.csv',   'INP-Former 20-cat')
abl4_ad_10  = safe_read_csv(
    f'{abl_results}/investigation4/real_iad/anomalydino_10cat_scores.csv', 'AnomalyDINO 10-cat')
abl4_ad_20  = safe_read_csv(
    f'{abl_results}/investigation4/real_iad/anomalydino_20cat_scores.csv', 'AnomalyDINO 20-cat')

print('\n=== MVTec AnomalyDINO Validation ===')
mvtec_sc_df = safe_read_csv(
    f'{results_path}/mvtec_anomalydino_singleclass.csv',
    'MVTec single-class')
mvtec_mc_df = safe_read_csv(
    f'{results_path}/mvtec_anomalydino_multiclass_comparison.csv',
    'MVTec multi-class comparison')

# Model dictionaries — only include models with data
df_dict_std = {k: v for k, v in {
    'AnomalyDINO': results_dino_std,
    'Dinomaly':    results_din_std,
    'INP-Former':  results_inp_std,
}.items() if v is not None}

df_dict_cv = {k: v for k, v in {
    'AnomalyDINO': results_dino_cv,
    'Dinomaly':    results_din_cv,
    'INP-Former':  results_inp_cv,
}.items() if v is not None}

print(f'\nStandard protocol models:   {list(df_dict_std.keys())}')
print(f'Cross-view protocol models: {list(df_dict_cv.keys())}')

## 3. Compute Pixel-Metrics 

In [ ]:
from adeval import EvalAccumulatorCuda
from PIL import Image as PILImage
import torch, torch.nn.functional as F, gc, os, numpy as np
from pathlib import Path

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
MAPS_DRIVE  = '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
MASKS_DRIVE = '/content/drive/MyDrive/datasets/realiad_512'

def resize_amap(amap, target_shape):
    amap_t = torch.tensor(amap).unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(
        amap_t,
        size=(target_shape[0], target_shape[1]),
        mode='bilinear',
        align_corners=False
    ).squeeze().numpy()
    return resized.astype(np.float32)

def compute_and_save_pixel_metrics(scores_df, model_name, protocol='standard'):
    maps_root  = MAPS_DRIVE
    masks_root = MASKS_DRIVE

    # Use full score range from all images for bounds
    score_min = float(scores_df['image_score'].min())
    score_max = float(scores_df['image_score'].max())
    print(f'{model_name} ({protocol})')
    print(f'Score bounds: [{score_min:.4f}, {score_max:.4f}]')

    # Build directory listing once per category
    category_map_files = {}
    for cat in scores_df['category'].unique():
        cat_dir = os.path.join(maps_root, protocol, model_name, cat)
        if os.path.exists(cat_dir):
            category_map_files[cat] = set(os.listdir(cat_dir))
        else:
            category_map_files[cat] = set()

    def make_acc():
        return EvalAccumulatorCuda(
            estimated_score_lower=score_min,
            estimated_score_upper=score_max,
            estimated_anomap_lower=score_min,
            estimated_anomap_upper=score_max,
            skip_pixel_aupro=False,
        )

    agg_acc  = make_acc()
    cat_accs = {}
    vp_accs  = {}
    def_accs = {}

    missing_maps  = 0
    missing_masks = 0
    processed     = 0

    for _, row in scores_df.iterrows():
        stem     = os.path.splitext(os.path.basename(row['image_path']))[0]
        cat      = row['category']
        vp       = row['viewpoint']
        def_type = row['defect_type']
        label    = int(row['label'])
        score    = float(row['image_score'])

        # Image-level — add for every image
        score_t = torch.tensor(score)
        label_t = torch.tensor(label)

        agg_acc.add_image(score=score_t, gtlabel=label_t)

        if cat not in cat_accs:
            cat_accs[cat] = make_acc()
        cat_accs[cat].add_image(score=score_t, gtlabel=label_t)

        if vp not in vp_accs:
            vp_accs[vp] = make_acc()
        vp_accs[vp].add_image(score=score_t, gtlabel=label_t)

        if def_type not in def_accs:
            def_accs[def_type] = make_acc()
        def_accs[def_type].add_image(score=score_t, gtlabel=label_t)

        # Pixel-level — only for anomalous images with saved maps
        if label == 1:
            fname    = f'{stem}.npz'
            npz_path = os.path.join(maps_root, protocol, model_name, cat, fname)
            mask_path = str(row.get('mask_path', '')).replace(
                '/content/realiad_512', masks_root)

            if fname not in category_map_files.get(cat, set()):
                missing_maps += 1
            elif not os.path.exists(mask_path):
                missing_masks += 1
            else:
                try:
                    data = np.load(npz_path)
                    amap = data['anomaly_map'].astype(np.float32)
                    mask = np.array(PILImage.open(mask_path).convert('L'))
                    mask = (mask > 127).astype(np.uint8)

                    if amap.shape != mask.shape:
                        amap = resize_amap(amap, mask.shape)

                    amap_t = torch.tensor(amap).unsqueeze(0).to(DEVICE)
                    mask_t = torch.tensor(mask).unsqueeze(0).to(DEVICE)

                    agg_acc.add_anomap_batch(amap_t, mask_t)
                    cat_accs[cat].add_anomap_batch(amap_t, mask_t)
                    vp_accs[vp].add_anomap_batch(amap_t, mask_t)
                    def_accs[def_type].add_anomap_batch(amap_t, mask_t)

                    del amap_t, mask_t

                except Exception as e:
                    missing_maps += 1
                    continue

        processed += 1
        if processed % 10000 == 0:
            print(f'  {processed}/{len(scores_df)} processed...')

    print(f'Done. Total: {processed}, '
          f'Missing maps: {missing_maps}, Missing masks: {missing_masks}')

    def extract(acc):
        try:
            res = acc.summary()
            return {
                'P-AUROC': round(res.get('p_auroc', float('nan')), 4),
                'P-AUPR':  round(res.get('p_aupr',  float('nan')), 4),
                'AUPRO':   round(res.get('p_aupro', float('nan')), 4),
            }
        except Exception as e:
            print(f'  Warning: summary failed — {e}')
            return {
                'P-AUROC': float('nan'),
                'P-AUPR':  float('nan'),
                'AUPRO':   float('nan'),
            }

    agg = extract(agg_acc)
    print(f'Aggregate — P-AUROC: {agg["P-AUROC"]}, '
          f'P-AUPR: {agg["P-AUPR"]}, AUPRO: {agg["AUPRO"]}')

    cat_rows = [{'category':    k, **extract(v)}
                for k, v in sorted(cat_accs.items())]
    vp_rows  = [{'viewpoint':   k, **extract(v)}
                for k, v in sorted(vp_accs.items())]
    def_rows = [{'defect_type': k, **extract(v)}
                for k, v in sorted(def_accs.items())]

    prefix = f'{results_path}/pixel_{protocol}_{model_name.lower().replace("-", "")}'
    pd.DataFrame([{'Model': model_name, **agg}]).to_csv(
        f'{prefix}_aggregate.csv', index=False)
    pd.DataFrame(cat_rows).to_csv(f'{prefix}_per_category.csv', index=False)
    pd.DataFrame(vp_rows).to_csv(f'{prefix}_per_viewpoint.csv', index=False)
    pd.DataFrame(def_rows).to_csv(f'{prefix}_per_defect.csv', index=False)
    print(f'Saved: {prefix}_*.csv')

    return {
        'aggregate':   agg,
        'category':    pd.DataFrame(cat_rows),
        'viewpoint':   pd.DataFrame(vp_rows),
        'defect_type': pd.DataFrame(def_rows),
    }

print('Pixel metric functions ready')

In [ ]:
# Dinomaly — Standard Protocol Pixel Metrics
print('=' * 60)
print('Dinomaly — Standard Protocol Pixel Metrics')
print('=' * 60)
din_pixel = compute_and_save_pixel_metrics(results_din_std, 'Dinomaly')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# INP-Former — Standard Protocol Pixel Metrics
print('=' * 60)
print('INP-Former — Standard Protocol Pixel Metrics')
print('=' * 60)
inp_pixel = compute_and_save_pixel_metrics(results_inp_std, 'INP-Former')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# AnomalyDINO — Standard Protocol Pixel Metrics
print('=' * 60)
print('AnomalyDINO — Standard Protocol Pixel Metrics')
print('=' * 60)
ad_pixel = compute_and_save_pixel_metrics(results_dino_std, 'AnomalyDINO')
gc.collect()
torch.cuda.empty_cache()

## 4. Anomaly Map Loading Helpers

In [ ]:
def load_anomaly_map(image_path, model_name, maps_root, protocol='standard'):
    """
    Load anomaly map and score for a given image, model, and protocol.
    Returns (anomaly_map H×W, score) or (None, None) if not found.
    """
    stem     = os.path.splitext(os.path.basename(image_path))[0]
    parts    = image_path.replace('\\', '/').split('/')
    category = parts[-3] if len(parts) >= 3 else 'unknown'
    npz_path = os.path.join(
        maps_root, protocol, model_name, category, f'{stem}.npz')
    if not os.path.exists(npz_path):
        return None, None
    data = np.load(npz_path)
    return data['anomaly_map'], float(data['anomaly_score'])


def load_maps_for_image(image_path, models, maps_root, protocol='standard'):
    """Load anomaly maps for all specified models for a given image."""
    result = {}
    for model_name in models:
        amap, score = load_anomaly_map(
            image_path, model_name, maps_root, protocol)
        if amap is not None:
            result[model_name] = (amap, score)
    return result

print('Map loading helpers ready')


## 5. Standard Protocol — Raw Numbers

In [ ]:
if not df_dict_std:
    print('No standard protocol data available')
else:
    print('=' * 70)
    print('STANDARD PROTOCOL — Aggregate: I-AUROC and S-AUROC')
    print('=' * 70)

    rows = []
    for model, df in df_dict_std.items():
        rows.append({
            'Model':    model,
            'Paradigm': {'AnomalyDINO': 'Memory-Based',
                         'Dinomaly':    'Reconstruction-Based',
                         'INP-Former':  'Prototype-Based'}[model],
            'I-AUROC':  round(compute_i_auroc(df), 4),
            'S-AUROC':  round(compute_s_auroc(df), 4),
        })

    std_agg = pd.DataFrame(rows)
    print(std_agg.to_string(index=False))
    std_agg.to_csv(f'{results_path}/summary_standard_aggregate.csv', index=False)
    print('\nSaved: summary_standard_aggregate.csv')

In [ ]:
if df_dict_std:
    print('=' * 70)
    print('STANDARD PROTOCOL — Per-Category I-AUROC')
    print('=' * 70)

    cat_dfs = []
    for model, df in df_dict_std.items():
        cat_auroc = df.groupby('category').apply(
            compute_i_auroc).reset_index()
        cat_auroc.columns = ['category', model]
        cat_dfs.append(cat_auroc.set_index('category'))

    cat_df     = pd.concat(cat_dfs, axis=1).reset_index()
    model_cols = list(df_dict_std.keys())
    cat_df['Mean'] = cat_df[model_cols].mean(axis=1).round(4)
    cat_df = cat_df.sort_values('category').reset_index(drop=True)

    mean_row = {'category': 'MEAN'}
    for m in model_cols:
        mean_row[m] = round(cat_df[m].mean(), 4)
    mean_row['Mean'] = round(cat_df['Mean'].mean(), 4)
    cat_df = pd.concat(
        [cat_df, pd.DataFrame([mean_row])], ignore_index=True)

    print(cat_df.round(4).to_string(index=False))
    cat_df.to_csv(
        f'{results_path}/summary_standard_per_category.csv', index=False)
    print('\nSaved: summary_standard_per_category.csv')

In [ ]:
if df_dict_std:
    print('=' * 70)
    print('STANDARD PROTOCOL — Per-Viewpoint I-AUROC')
    print('=' * 70)

    vp_dfs = []
    for model, df in df_dict_std.items():
        vp = df.groupby('viewpoint').apply(
            compute_i_auroc).reset_index()
        vp.columns = ['viewpoint', model]
        vp_dfs.append(vp.set_index('viewpoint'))

    vp_df = pd.concat(vp_dfs, axis=1).reset_index().sort_values('viewpoint')
    print(vp_df.round(4).to_string(index=False))
    vp_df.to_csv(
        f'{results_path}/summary_standard_per_viewpoint.csv', index=False)
    print('\nSaved: summary_standard_per_viewpoint.csv')

In [ ]:
if df_dict_std:
    print('=' * 70)
    print('STANDARD PROTOCOL — Per-Defect-Type I-AUROC')
    print('=' * 70)

    def_dfs = []
    for model, df in df_dict_std.items():
        def_auroc = df.groupby('defect_type').apply(
            compute_i_auroc).reset_index()
        def_auroc.columns = ['defect_type', model]
        def_dfs.append(def_auroc.set_index('defect_type'))

    def_df = pd.concat(def_dfs, axis=1).reset_index()
    def_df = def_df.sort_values(
        list(df_dict_std.keys())[0]).reset_index(drop=True)
    print(def_df.round(4).to_string(index=False))
    def_df.to_csv(
        f'{results_path}/summary_standard_per_defect.csv', index=False)
    print('\nSaved: summary_standard_per_defect.csv')

In [ ]:
if df_dict_std:
    print('=' * 70)
    print('STANDARD PROTOCOL — Score Separation Check')
    print('=' * 70)

    for model, df in df_dict_std.items():
        n_mean = df[df['label'] == 0]['image_score'].mean()
        a_mean = df[df['label'] == 1]['image_score'].mean()
        s_range = df['image_score'].max() - df['image_score'].min()
        print(f'\n{model}:')
        print(f'  Normal mean:  {n_mean:.4f}')
        print(f'  Anomaly mean: {a_mean:.4f}')
        print(f'  Separation:   {a_mean - n_mean:.4f}')
        print(f'  Score range:  {s_range:.4f}')

In [ ]:
if df_dict_std:
    print('=' * 70)
    print('STANDARD PROTOCOL — WGA Summary')
    print('=' * 70)
    print_wga_summary(df_dict_std)

In [ ]:
if df_dict_std:
    print('=' * 70)
    print('STANDARD PROTOCOL — Disagreement Analysis (per category)')
    print('=' * 70)

    cat_auroc_all = {}
    for model, df in df_dict_std.items():
        cat_auroc_all[model] = df.groupby('category').apply(
            compute_i_auroc)

    disag = pd.DataFrame(cat_auroc_all).reset_index()
    disag.columns = ['category'] + list(df_dict_std.keys())
    model_list = list(df_dict_std.keys())

    disag['max_disagreement'] = (
        disag[model_list].max(axis=1) - disag[model_list].min(axis=1))
    disag = disag.sort_values('max_disagreement', ascending=False)

    print('Top 10 highest-disagreement categories:')
    print(disag.head(10).round(4).to_string(index=False))
    disag.to_csv(
        f'{results_path}/summary_standard_disagreement.csv', index=False)
    print('\nSaved: summary_standard_disagreement.csv')


## 6. Cross-View Protocol — Raw Numbers

In [ ]:
if not df_dict_cv:
    print('No cross-view data available')
else:
    print('=' * 70)
    print('CROSS-VIEW PROTOCOL — Aggregate Summary')
    print('=' * 70)

    cv_rows = []
    for model, df in df_dict_cv.items():
        if df_dict_std.get(model) is None:
            print(f'  WARNING: no standard result for {model}')
            continue
        i_std = compute_i_auroc(df_dict_std[model])
        i_cv  = compute_i_auroc(df)
        s_cv  = compute_s_auroc(df)
        deg   = compute_degradation_ratio(i_std, i_cv)
        cv_rows.append({
            'Model':              model,
            'I-AUROC Standard':   round(i_std, 4),
            'I-AUROC Cross-View': round(i_cv,  4),
            'S-AUROC Cross-View': round(s_cv,  4),
            'Degradation (%)':    round(deg,   2),
        })

    cv_agg = pd.DataFrame(cv_rows)
    print(cv_agg.to_string(index=False))
    cv_agg.to_csv(
        f'{results_path}/summary_crossview_aggregate.csv', index=False)
    print('\nSaved: summary_crossview_aggregate.csv')

In [ ]:
if df_dict_cv:
    print('=' * 70)
    print('CROSS-VIEW — Per-Viewpoint I-AUROC (Standard vs Cross-View)')
    print('=' * 70)

    delta_rows = []
    for model in df_dict_cv:
        if df_dict_std.get(model) is None:
            continue
        std_vp = df_dict_std[model].groupby('viewpoint').apply(
            compute_i_auroc)
        cv_vp  = df_dict_cv[model].groupby('viewpoint').apply(
            compute_i_auroc)
        merged = pd.DataFrame(
            {'Standard': std_vp, 'Cross-View': cv_vp})
        merged['Delta'] = (
            merged['Cross-View'] - merged['Standard']).round(4)
        print(f'\n{model}:')
        print(merged.round(4).to_string())
        delta_rows.append((merged['Delta']).rename(model))

    if delta_rows:
        delta_vp = pd.concat(delta_rows, axis=1).reset_index()
        delta_vp.to_csv(
            f'{results_path}/summary_crossview_per_viewpoint_delta.csv',
            index=False)
        print('\nSaved: summary_crossview_per_viewpoint_delta.csv')

In [ ]:
if df_dict_cv:
    print('=' * 70)
    print('CROSS-VIEW — Per-Category I-AUROC Delta')
    print('=' * 70)

    delta_cat_rows = []
    for model in df_dict_cv:
        if df_dict_std.get(model) is None:
            continue
        std_cat = df_dict_std[model].groupby('category').apply(
            compute_i_auroc)
        cv_cat  = df_dict_cv[model].groupby('category').apply(
            compute_i_auroc)
        delta   = (cv_cat - std_cat).rename(model)
        delta_cat_rows.append(delta)

    if delta_cat_rows:
        delta_cat = pd.concat(delta_cat_rows, axis=1).reset_index()
        delta_cat.columns = (
            ['category'] + [r.name for r in delta_cat_rows])
        model_cols = [r.name for r in delta_cat_rows]
        delta_cat['worst_model'] = delta_cat[model_cols].idxmin(axis=1)
        delta_cat = delta_cat.sort_values(model_cols[0])
        print('Negative = performance drop under viewpoint shift:')
        print(delta_cat.round(4).to_string(index=False))
        delta_cat.to_csv(
            f'{results_path}/summary_crossview_per_category_delta.csv',
            index=False)
        print('\nSaved: summary_crossview_per_category_delta.csv')


## 7. Ablation Investigations — Raw Numbers

In [ ]:
print('=' * 70)
print('INVESTIGATION 1 — AnomalyDINO 2x2 Factorial Design')
print('=' * 70)

conditions = {
    'A: Multi-class, Multi-view (Standard)': abl1_cond_a,
    'B: Single-class, Multi-view':           abl1_cond_b,
    'C: Multi-class, Single-view':           abl1_cond_c,
    'D: Single-class, Single-view':          abl1_cond_d,
}

fact_rows = []
for cond_name, df in conditions.items():
    if df is None:
        print(f'  MISSING: {cond_name}')
        fact_rows.append({'Condition': cond_name, 'I-AUROC': None})
    else:
        fact_rows.append({
            'Condition': cond_name,
            'I-AUROC':   round(compute_i_auroc(df), 4)
        })

fact_df = pd.DataFrame(fact_rows)
print(fact_df.to_string(index=False))

vals = {r['Condition'][0]: r['I-AUROC']
        for r in fact_rows if r['I-AUROC'] is not None}
if 'A' in vals and 'C' in vals:
    print(f'\nViewpoint effect (A minus C): {round(vals["A"] - vals["C"], 4)}')
if 'A' in vals and 'B' in vals:
    print(f'Category effect  (A minus B): {round(vals["A"] - vals["B"], 4)}')
if 'D' in vals:
    print(f'Upper bound      (Cond D):    {vals["D"]}')

fact_df.to_csv(
    f'{results_path}/summary_investigation1_factorial.csv', index=False)
print('\nSaved: summary_investigation1_factorial.csv')

In [ ]:
print('INVESTIGATION 1 — Per-Category Breakdown Across Conditions')

cond_dfs = {
    'A': abl1_cond_a, 'B': abl1_cond_b,
    'C': abl1_cond_c, 'D': abl1_cond_d
}
per_cat_conds = {}
for cond, df in cond_dfs.items():
    if df is not None:
        per_cat_conds[f'Cond_{cond}'] = df.groupby('category').apply(
            compute_i_auroc)

if per_cat_conds:
    cond_cat = pd.DataFrame(per_cat_conds).reset_index()
    cond_cat.columns = ['category'] + list(per_cat_conds.keys())
    print(cond_cat.round(4).to_string(index=False))
    cond_cat.to_csv(
        f'{results_path}/summary_investigation1_per_category.csv',
        index=False)
    print('\nSaved: summary_investigation1_per_category.csv')
else:
    print('No condition data available')

In [ ]:
print('=' * 70)
print('INVESTIGATION 2 — Compute Equalisation')
print('=' * 70)

if (results_din_std is not None
        and results_inp_std is not None
        and abl2_inp_eq is not None):
    i_din_std = compute_i_auroc(results_din_std)
    i_inp_std = compute_i_auroc(results_inp_std)
    i_inp_eq  = compute_i_auroc(abl2_inp_eq)

    inv2 = pd.DataFrame([
        {'Model': 'Dinomaly',
         'Config': 'Standard (50K iter, ~800K passes)',
         'I-AUROC': round(i_din_std, 4)},
        {'Model': 'INP-Former',
         'Config': 'Standard (100 epochs, ~3.6M passes)',
         'I-AUROC': round(i_inp_std, 4)},
        {'Model': 'INP-Former',
         'Config': 'Equalised (22 epochs, ~800K passes)',
         'I-AUROC': round(i_inp_eq, 4)},
    ])
    print(inv2.to_string(index=False))
    print(f'\nGap at standard compute:  {round(i_inp_std - i_din_std, 4)}')
    print(f'Gap at equalised compute: {round(i_inp_eq  - i_din_std, 4)}')
    inv2.to_csv(
        f'{results_path}/summary_investigation2_compute.csv', index=False)
    print('\nSaved: summary_investigation2_compute.csv')
else:
    print('Missing data for Investigation 2')

In [ ]:
print('=' * 70)
print('INVESTIGATION 3 — Cross-View Volume Compensation (INP-Former)')
print('=' * 70)

if (results_inp_std is not None
        and results_inp_cv is not None
        and abl3_inp_comp is not None):
    i_std = compute_i_auroc(results_inp_std)
    i_cv  = compute_i_auroc(results_inp_cv)
    i_com = compute_i_auroc(abl3_inp_comp)

    inv3 = pd.DataFrame([
        {'Config': 'Standard — 100 epochs, all viewpoints',
         'I-AUROC': round(i_std, 4)},
        {'Config': 'Cross-view — 100 epochs',
         'I-AUROC': round(i_cv,  4)},
        {'Config': 'Cross-view — 250 epochs (compensated)',
         'I-AUROC': round(i_com, 4)},
    ])
    print(inv3.to_string(index=False))
    print(f'\nDegradation at 100 epochs: {round(i_std - i_cv,  4)}pp')
    print(f'Recovery at 250 epochs:    {round(i_com - i_cv,  4):+.4f}pp')
    inv3.to_csv(
        f'{results_path}/summary_investigation3_volume.csv', index=False)
    print('\nSaved: summary_investigation3_volume.csv')
else:
    print('Missing data for Investigation 3')

In [ ]:
print('=' * 70)
print('INVESTIGATION 4 — Category Scaling Aggregate')
print('=' * 70)

din_30 = compute_i_auroc(results_din_std)  if results_din_std  is not None else None
inp_30 = compute_i_auroc(abl2_inp_eq)      if abl2_inp_eq      is not None else None
ad_30  = compute_i_auroc(results_dino_std) if results_dino_std is not None else None
din_20 = compute_i_auroc(abl4_din_20)      if abl4_din_20      is not None else None
din_10 = compute_i_auroc(abl4_din_10)      if abl4_din_10      is not None else None
inp_20 = compute_i_auroc(abl4_inp_20)      if abl4_inp_20      is not None else None
inp_10 = compute_i_auroc(abl4_inp_10)      if abl4_inp_10      is not None else None
ad_20  = compute_i_auroc(abl4_ad_20)       if abl4_ad_20       is not None else None
ad_10  = compute_i_auroc(abl4_ad_10)       if abl4_ad_10       is not None else None

print(f'Dinomaly (Real-IAD, proportional iterations):')
print(f'  10 categories: {din_10}')
print(f'  20 categories: {din_20}')
print(f'  30 categories: {din_30}  (standard protocol)')
print(f'\nINP-Former (Real-IAD, 22 epochs):')
print(f'  10 categories: {inp_10}')
print(f'  20 categories: {inp_20}')
print(f'  30 categories: {inp_30}  (Investigation 2 baseline)')
print(f'\nAnomalyDINO (Real-IAD, 16-shot):')
print(f'  10 categories: {ad_10}')
print(f'  20 categories: {ad_20}')
print(f'  30 categories: {ad_30}  (standard protocol)')

inv4_agg = pd.DataFrame([
    {'Model': 'Dinomaly',    'n_categories': 10, 'i_auroc': din_10},
    {'Model': 'Dinomaly',    'n_categories': 20, 'i_auroc': din_20},
    {'Model': 'Dinomaly',    'n_categories': 30, 'i_auroc': din_30},
    {'Model': 'INP-Former',  'n_categories': 10, 'i_auroc': inp_10},
    {'Model': 'INP-Former',  'n_categories': 20, 'i_auroc': inp_20},
    {'Model': 'INP-Former',  'n_categories': 30, 'i_auroc': inp_30},
    {'Model': 'AnomalyDINO', 'n_categories': 10, 'i_auroc': ad_10},
    {'Model': 'AnomalyDINO', 'n_categories': 20, 'i_auroc': ad_20},
    {'Model': 'AnomalyDINO', 'n_categories': 30, 'i_auroc': ad_30},
])
inv4_agg.to_csv(
    f'{abl_results}/investigation4/investigation4_summary.csv', index=False)
print('\nSaved: investigation4_summary.csv')

In [ ]:
print('=' * 70)
print('INVESTIGATION 4 — Per-Category Scaling')
print('=' * 70)

if results_din_std is not None:
    all_cats = sorted(results_din_std['category'].unique())
    cats_10  = all_cats[:10]
    cats_20  = all_cats[:20]

    def per_cat(df):
        if df is None: return {}
        return df.groupby('category').apply(compute_i_auroc).to_dict()

    din_30_pc = per_cat(results_din_std)
    din_20_pc = per_cat(abl4_din_20)
    din_10_pc = per_cat(abl4_din_10)
    inp_30_pc = per_cat(abl2_inp_eq)
    inp_20_pc = per_cat(abl4_inp_20)
    inp_10_pc = per_cat(abl4_inp_10)
    ad_30_pc  = per_cat(results_dino_std)
    ad_20_pc  = per_cat(abl4_ad_20)
    ad_10_pc  = per_cat(abl4_ad_10)

    rows = []
    for cat in all_cats:
        group = ('10+20+30' if cat in cats_10
                 else '20+30' if cat in cats_20
                 else '30 only')
        row = {'category': cat, 'group': group}
        if cat in cats_10:
            row['Din_10'] = round(din_10_pc.get(cat, float('nan')), 4)
            row['INP_10'] = round(inp_10_pc.get(cat, float('nan')), 4)
            row['AD_10']  = round(ad_10_pc.get(cat,  float('nan')), 4)
        if cat in cats_20:
            row['Din_20'] = round(din_20_pc.get(cat, float('nan')), 4)
            row['INP_20'] = round(inp_20_pc.get(cat, float('nan')), 4)
            row['AD_20']  = round(ad_20_pc.get(cat,  float('nan')), 4)
        row['Din_30'] = round(din_30_pc.get(cat, float('nan')), 4)
        row['INP_30'] = round(inp_30_pc.get(cat, float('nan')), 4)
        row['AD_30']  = round(ad_30_pc.get(cat,  float('nan')), 4)
        rows.append(row)

    inv4_cat = pd.DataFrame(rows)

    print('Group 1 — First 10 categories (all three configurations):')
    print(inv4_cat[inv4_cat['group'] == '10+20+30'].to_string(index=False))
    print('\nGroup 2 — Next 10 categories (20 and 30 cat):')
    print(inv4_cat[inv4_cat['group'] == '20+30'].to_string(index=False))

    inv4_cat.to_csv(
        f'{results_path}/summary_investigation4_per_category.csv',
        index=False)
    print('\nSaved: summary_investigation4_per_category.csv')
else:
    print('Standard protocol data needed for category list')

In [ ]:
print('=' * 70)
print('MVTEC VALIDATION — AnomalyDINO Single vs Multi-class')
print('=' * 70)

if mvtec_mc_df is not None:
    cols = [c for c in [
        'Category', 'Single-class I-AUROC', 'SC Std',
        'Multi-class I-AUROC', 'Difference (MC - SC)']
        if c in mvtec_mc_df.columns]
    print(mvtec_mc_df[cols].to_string(index=False))
    print(f'\nMean single-class: {mvtec_mc_df["Single-class I-AUROC"].mean():.2f}%')
    print(f'Mean multi-class:  {mvtec_mc_df["Multi-class I-AUROC"].mean():.2f}%')
    print(f'Mean difference:   {mvtec_mc_df["Difference (MC - SC)"].mean():.2f}pp')
    mvtec_mc_df.to_csv(
        f'{results_path}/summary_mvtec_validation.csv', index=False)
    print('\nSaved: summary_mvtec_validation.csv')
else:
    print('MVTec comparison data not available')


## 8. Combined Summary Table

In [ ]:
print('=' * 70)
print('COMBINED RESULTS SUMMARY — All Protocols')
print('=' * 70)

combined_rows = []
for model in ['AnomalyDINO', 'Dinomaly', 'INP-Former']:
    if df_dict_std.get(model) is None:
        continue
    i_std = compute_i_auroc(df_dict_std[model])
    s_std = compute_s_auroc(df_dict_std[model])
    if df_dict_cv.get(model) is not None:
        i_cv = compute_i_auroc(df_dict_cv[model])
        deg  = compute_degradation_ratio(i_std, i_cv)
    else:
        i_cv, deg = None, None

    combined_rows.append({
        'Model':           model,
        'Paradigm':        {'AnomalyDINO': 'Memory-Based',
                            'Dinomaly':    'Reconstruction-Based',
                            'INP-Former':  'Prototype-Based'}[model],
        'I-AUROC Std':     round(i_std, 4),
        'S-AUROC Std':     round(s_std, 4),
        'I-AUROC CV':      round(i_cv, 4) if i_cv is not None else 'N/A',
        'Degradation (%)': round(deg, 2)  if deg  is not None else 'N/A',
    })

combined_df = pd.DataFrame(combined_rows)
print(combined_df.to_string(index=False))
combined_df.to_csv(f'{results_path}/summary_combined.csv', index=False)
print('\nSaved: summary_combined.csv')


## 9. Figure Cells

Run all data cells above and review the numbers before building figures.
Figure cells will be added here once data distributions are known.

In [ ]:
print('Figure cells to be added after data review')
print(f'All figures will be saved to: {figures_path}')


## 10. File Index

In [ ]:
print('=' * 70)
print('GENERATED FILES')
print('=' * 70)

expected = [
    f'{results_path}/summary_standard_aggregate.csv',
    f'{results_path}/summary_standard_per_category.csv',
    f'{results_path}/summary_standard_per_viewpoint.csv',
    f'{results_path}/summary_standard_per_defect.csv',
    f'{results_path}/summary_standard_disagreement.csv',
    f'{results_path}/summary_crossview_aggregate.csv',
    f'{results_path}/summary_crossview_per_viewpoint_delta.csv',
    f'{results_path}/summary_crossview_per_category_delta.csv',
    f'{results_path}/summary_investigation1_factorial.csv',
    f'{results_path}/summary_investigation1_per_category.csv',
    f'{results_path}/summary_investigation2_compute.csv',
    f'{results_path}/summary_investigation3_volume.csv',
    f'{results_path}/summary_investigation4_per_category.csv',
    f'{abl_results}/investigation4/investigation4_summary.csv',
    f'{results_path}/summary_mvtec_validation.csv',
    f'{results_path}/summary_combined.csv',
]

for fpath in expected:
    status = 'OK     ' if Path(fpath).exists() else 'MISSING'
    print(f'  [{status}] {Path(fpath).name}')